In [ ]:
!unzip multilabel-classification-dataset.zip -d /content/dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install scikit-multilearn


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW # Import AdamW from torch.optim
from sklearn.metrics import f1_score
# from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit # Removing due to compatibility issues
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
# Load CSV
df = pd.read_csv("/content/dataset/train.csv")

# Combine Title + Abstract
df["text"] = df["TITLE"].astype(str) + " " + df["ABSTRACT"].astype(str)

# Define labels
LABEL_NAMES = ["Computer Science", "Physics", "Mathematics", "Statistics", "Quantitative Biology", "Quantitative Finance"]
labels = df[LABEL_NAMES].values
texts = df["text"].tolist()

print("Dataset size:", len(df))


In [ ]:
from transformers import T5Tokenizer, T5ForSequenceClassification

model_name = "t5-small"  # or "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForSequenceClassification.from_pretrained(model_name, num_labels=6)


In [ ]:
import pandas as pd

# Load your train and test CSVs
train_df = pd.read_csv("/content/dataset/train.csv")
test_df = pd.read_csv("/content/dataset/train.csv")

# If you want a validation split from train:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)


In [ ]:
train_texts = (train_df["TITLE"] + " " + train_df["ABSTRACT"]).tolist()
val_texts = (val_df["TITLE"] + " " + val_df["ABSTRACT"]).tolist()

train_labels = train_df.iloc[:, 2:].values  # adjust if label columns differ
val_labels = val_df.iloc[:, 2:].values

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")


In [ ]:
train_df.iloc[:, 2:].head()


In [ ]:
print(train_df.columns)


In [ ]:
train_labels = train_df.iloc[:, 3:].astype(float).values
val_labels = val_df.iloc[:, 3:].astype(float).values


In [ ]:
from torch.utils.data import TensorDataset
import torch

train_dataset = TensorDataset(
    torch.tensor(train_encodings["input_ids"]),
    torch.tensor(train_encodings["attention_mask"]),
    torch.tensor(train_labels, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(val_encodings["input_ids"]),
    torch.tensor(val_encodings["attention_mask"]),
    torch.tensor(val_labels, dtype=torch.float32)
)


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# assuming you already have these:
# train_encodings, val_encodings, train_labels, val_labels

train_dataset = TensorDataset(
    torch.tensor(train_encodings["input_ids"]),
    torch.tensor(train_encodings["attention_mask"]),
    torch.tensor(train_labels, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(val_encodings["input_ids"]),
    torch.tensor(val_encodings["attention_mask"]),
    torch.tensor(val_labels, dtype=torch.float32)
)

#  create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4)


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results_t5",
    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# =========================================================
# LOSS & OPTIMIZER
# =========================================================
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

# =========================================================
# TRAINING SETUP
# =========================================================
num_epochs = 10

train_losses, val_losses = [], []

train_accuracies, val_accuracies = [], []
train_labelwise_accuracies, val_labelwise_accuracies = [], []

train_precisions, val_precisions = [], []
train_recalls, val_recalls = [], []
train_f1s, val_f1s = [], []

# =========================================================
# TRAINING LOOP
# =========================================================
for epoch in range(num_epochs):

    # ===================== TRAIN =====================
    model.train()
    total_train_loss = 0
    train_preds, train_labels = [], []

    for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()

        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        logits = outputs.logits
        preds = torch.sigmoid(logits).detach().cpu().numpy()

        train_preds.append(preds)
        train_labels.append(labels.cpu().numpy())

    avg_train_loss = total_train_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)

    train_preds = np.vstack(train_preds)
    train_labels = np.vstack(train_labels)
    train_preds_bin = (train_preds > 0.5).astype(int)

    # Subset Accuracy (Exact Match)
    train_acc = accuracy_score(train_labels, train_preds_bin)

    # Label-wise Overall Accuracy
    train_labelwise_acc = (train_labels == train_preds_bin).mean()

    train_prec = precision_score(train_labels, train_preds_bin, average="micro", zero_division=0)
    train_rec = recall_score(train_labels, train_preds_bin, average="micro", zero_division=0)
    train_f1 = f1_score(train_labels, train_preds_bin, average="micro", zero_division=0)

    train_accuracies.append(train_acc)
    train_labelwise_accuracies.append(train_labelwise_acc)
    train_precisions.append(train_prec)
    train_recalls.append(train_rec)
    train_f1s.append(train_f1)

    # ===================== VALIDATION =====================
    model.eval()
    total_val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc="Validating"):
            input_ids, attention_mask, labels = [b.to(device) for b in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_val_loss += loss.item()

            logits = outputs.logits
            preds = torch.sigmoid(logits).cpu().numpy()

            val_preds.append(preds)
            val_labels.append(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_losses.append(avg_val_loss)

    val_preds = np.vstack(val_preds)
    val_labels = np.vstack(val_labels)
    val_preds_bin = (val_preds > 0.5).astype(int)

    # 🔹 Subset Accuracy (Exact Match)
    val_acc = accuracy_score(val_labels, val_preds_bin)

    # 🔹 NEW: Label-wise Overall Accuracy
    val_labelwise_acc = (val_labels == val_preds_bin).mean()

    val_prec = precision_score(val_labels, val_preds_bin, average="micro", zero_division=0)
    val_rec = recall_score(val_labels, val_preds_bin, average="micro", zero_division=0)
    val_f1 = f1_score(val_labels, val_preds_bin, average="micro", zero_division=0)

    val_accuracies.append(val_acc)
    val_labelwise_accuracies.append(val_labelwise_acc)
    val_precisions.append(val_prec)
    val_recalls.append(val_rec)
    val_f1s.append(val_f1)

    # ===================== LOGGING =====================
    print(
        f"Epoch {epoch+1}/{num_epochs} "
        f"Train | Loss: {avg_train_loss:.4f} | "
        f"SubsetAcc: {train_acc:.4f} | "
        f"LabelAcc: {train_labelwise_acc:.4f} | "
        f"Prec: {train_prec:.4f} | "
        f"Rec: {train_rec:.4f} | F1: {train_f1:.4f} "
        f"Val | Loss: {avg_val_loss:.4f} | "
        f"SubsetAcc: {val_acc:.4f} | "
        f"LabelAcc: {val_labelwise_acc:.4f} | "
        f"Prec: {val_prec:.4f} | "
        f"Rec: {val_rec:.4f} | F1: {val_f1:.4f}"
    )

# =========================================================
# SAVE FINAL MODEL
# =========================================================
torch.save(model.state_dict(), "t5_model_epoch10.pt")

print("\nTraining complete  Model trained for full 10 epochs.")

In [ ]:
save_path = "/content/drive/MyDrive/T5_updated"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f" Model and tokenizer saved to {save_path}")


In [ ]:
from transformers import T5ForSequenceClassification, T5Tokenizer

load_path = "/content/drive/MyDrive/T5_updated"

model = T5ForSequenceClassification.from_pretrained(load_path)
tokenizer = T5Tokenizer.from_pretrained(load_path)

print(" Model and tokenizer loaded successfully!")


In [ ]:
model.to(device)


In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#  Load best model
model.load_state_dict(torch.load("best_t5_model.pt"))
model.eval()

#  Label names
label_names = [
    "Computer Science",
    "Physics",
    "Mathematics",
    "Statistics",
    "Quantitative Biology",
    "Quantitative Finance"
]

all_preds, all_labels = [], []

#  Validation Loop
with torch.no_grad():
    for batch in tqdm(val_dataloader, desc="Evaluating"):
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.cpu().numpy())

#  Convert to arrays
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)
all_preds_bin = (all_preds > 0.5).astype(int)

#  Overall metrics
acc = accuracy_score(all_labels, all_preds_bin)
micro_f1 = f1_score(all_labels, all_preds_bin, average="micro")
macro_f1 = f1_score(all_labels, all_preds_bin, average="macro")
micro_precision = precision_score(all_labels, all_preds_bin, average="micro")
micro_recall = recall_score(all_labels, all_preds_bin, average="micro")
hamming = np.mean(all_labels != all_preds_bin)

print("\n Overall Validation Metrics")
print(f"Accuracy Score: {acc:.6f}")
print(f"Micro F1 Score: {micro_f1:.6f}")
print(f"Macro F1 Score: {macro_f1:.6f}")
print(f"Micro Precision: {micro_precision:.6f}")
print(f"Micro Recall: {micro_recall:.6f}")
print(f"Hamming Loss: {hamming:.6f}")

#  Per-label accuracies
print("\n Per-label Validation Accuracy:")
per_label_accuracies = []
for i, label in enumerate(label_names):
    label_acc = (all_labels[:, i] == all_preds_bin[:, i]).mean()
    per_label_accuracies.append(label_acc)
    print(f"{label}: {label_acc:.6f}")

avg_label_acc = np.mean(per_label_accuracies)
print(f"\nAverage per-label accuracy = {avg_label_acc:.6f}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
save_path = "/content/drive/MyDrive/t5_finetuned_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f" Model and tokenizer saved to: {save_path}")
